# 3단계: 분류기 개선 (작은 신경망, 프레임 단위 학습)

1·2단계에서 저장한 임베딩(`embeddings_v2.npz`)을 드라이브에서 읽어서 쓰기 때문에, YAMNet을 다시 돌릴 필요가 없습니다.
**GPU는 필요 없습니다.** 기본(CPU) 런타임으로 실행하세요. (몇 분 걸립니다)

달라지는 점
- 클립 평균에 로지스틱 회귀 → **프레임(약 1초) 단위로 작은 신경망(MLP)** 학습
- 판정 규칙을 **"N프레임 연속으로 점수가 임계값 이상이면 위급"** 으로 바꿔서 오경보를 줄이는 효과 확인
- 임계값·N을 바꿔 가며 **위급 재현율 vs 오경보** 표 만들기

위에서부터 순서대로 실행하고, **5번, 6번 항목의 출력**을 복사해서 알려 주세요.

## 0. 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 임베딩 읽기

In [ ]:
import glob, os
import numpy as np
import pandas as pd

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    assert hits, f"{name}을 못 찾았어요."
    return hits[0]

EMB = find("embeddings_v2.npz")
OUT_DIR = os.path.dirname(EMB)
d = np.load(EMB)
X, clip_idx = d["X"].astype(np.float32), d["clip_idx"]
df = pd.read_csv(find("dataset_v2.csv"), encoding="utf-8-sig")

y_clip = (df["label"] == "urgent").astype(int).values
is_tr = (df["split"] == "train").values
cls = df["class"].values

starts = np.r_[0, np.flatnonzero(np.diff(clip_idx)) + 1]
ends = np.r_[starts[1:], len(clip_idx)]
assert len(starts) == len(df)
frames = ends - starts
print("프레임", X.shape, "| 클립", len(df), "| 학습 클립", int(is_tr.sum()), "| 검증 클립", int((~is_tr).sum()))

## 2. 학습 준비
- 정답은 프레임마다 "그 클립의 라벨"을 붙입니다.
- 긴 정상 클립(프레임 20개)이 짧은 위급 클립(프레임 약 7개)보다 학습을 지배하지 않도록, **클립마다 같은 비중**으로 가중치를 줍니다. 위급:정상 클립 수 차이도 같이 보정합니다.

In [ ]:
y_frame = y_clip[clip_idx].astype(np.float32)
tr_frame = is_tr[clip_idx]

n_urg = int(y_clip[is_tr].sum()); n_nor = int(is_tr.sum()) - n_urg
class_w = {1: 0.5 * is_tr.sum() / n_urg, 0: 0.5 * is_tr.sum() / n_nor}
w_frame = np.array([class_w[int(l)] for l in y_clip])[clip_idx] / frames[clip_idx]
w_frame = (w_frame / w_frame[tr_frame].mean()).astype(np.float32)
print("학습 클립: 위급", n_urg, "/ 정상", n_nor)

## 3. 학습 (작은 신경망)
검증 손실이 4번 연속 좋아지지 않으면 멈추고 가장 좋았던 상태로 되돌립니다.

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024,)),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
model.fit(X[tr_frame], y_frame[tr_frame], sample_weight=w_frame[tr_frame],
          validation_data=(X[~tr_frame], y_frame[~tr_frame], w_frame[~tr_frame]),
          epochs=40, batch_size=512, callbacks=[es], verbose=2)
p = model.predict(X, batch_size=4096, verbose=0).ravel()   # 프레임마다 '위급일 확률'
print("파라미터 수:", model.count_params())

## 4. 판정 규칙 함수
클립의 점수 = "**N프레임 연속**으로 확률이 얼마 이상이었나" (N=1이면 한 프레임이라도 높으면 위급).
임계값을 이 점수와 비교해서 위급 여부를 정합니다.

In [ ]:
def consec(p, N):
    out = np.zeros(len(starts))
    for i, (s, e) in enumerate(zip(starts, ends)):
        q = p[s:e]
        if len(q) >= N:
            out[i] = np.lib.stride_tricks.sliding_window_view(q, N).min(axis=1).max()
    return out

va = ~is_tr
urgent_classes = ["10.낙상", "14.도움요청", "7.응급의료"]
normal_hours = df.loc[va & (y_clip == 0), "duration_sec"].sum() / 3600
print(f"Validation 정상 소리 총 길이: {normal_hours:.2f}시간")

## 5. 클래스별 맞힌 비율 (N=1, 임계값 0.5)
이전 베이스라인(클립 평균 + 로지스틱 회귀)은 낙상 0.980, 도움요청 0.977, 응급의료 0.903, 실내 0.970, 실외 0.995였습니다.

In [ ]:
sc1 = consec(p, 1)
df["ok"] = (sc1 >= 0.5) == (y_clip == 1)
print(df.groupby(["class", "split"])["ok"].mean().unstack().round(3)
        .rename(columns={"train": "학습 데이터", "val": "Validation"}))

## 6. 임계값·N별 "위급 재현율 vs 오경보"
- 재현율: Validation의 위급 클립 중 알림이 울린 비율 (클래스별도 표시)
- 정상오경보: Validation의 정상 클립 중 알림이 울린 비율
- 시간당 오경보(약): 정상 클립에서 알림이 울린 클립 수 ÷ 정상 소리 총 시간. **AI Hub 정상 소리 기준이라 낙관적인 값**입니다. (일상 대화·TV는 아직 없음)

In [ ]:
rows = []
for N in (1, 2, 3):
    sc = consec(p, N)
    for thr in (0.5, 0.7, 0.8, 0.9, 0.95):
        alarm = sc >= thr
        row = {"N": N, "임계값": thr,
               "위급재현율": alarm[va & (y_clip == 1)].mean()}
        for c in urgent_classes:
            row[c] = alarm[va & (cls == c)].mean()
        fa = alarm[va & (y_clip == 0)]
        row["정상오경보"] = fa.mean()
        row["시간당오경보(약)"] = fa.sum() / normal_hours
        rows.append(row)
pd.set_option("display.width", 200)
print(pd.DataFrame(rows).round(3).to_string(index=False))

## 7. 모델 저장
드라이브의 같은 폴더에 저장됩니다. 다음 단계(Pi용 변환)에서 이 파일을 씁니다.

In [ ]:
model.save(f"{OUT_DIR}/classifier_mlp_v1.keras")
print("저장 완료:", f"{OUT_DIR}/classifier_mlp_v1.keras")